# 3. Command：跳转与更新二合一

前面 Node 都是「返回 dict 更新 State」，跳转交给边。`Command` 让 Node 在返回时**同时**写 State 和指定下一跳：

- `Command(goto=..., update=...)`：`goto` 指定下一个节点（覆盖图上的边），`update` 是 State 更新（正常走 reducer 合并）
- 适用场景：agent 手写循环（模型输出 Command 直接指定下一步并携带数据）、「边跑边决定」的复杂控制流
- `Command` 从 `langgraph.types` 导入

> 等价关系：`return Command(goto="b", update={...})` ≈ `return {...}` + 条件边路由到 b。Command 把两件事合成一个返回值，让 Node 自己决定走向。

In [ ]:
# 示例：Command —— check 节点同时更新 State 并指定下一跳（改写第 2 节的自纠错循环）
# 对比第 2 节案例：循环原来靠「stop 字段 + route 函数 + add_conditional_edges」实现，
# 这里 check_node 直接返回 Command(goto=..., update=...)：路由函数、stop 字段、条件边都不再需要

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    response: str


def llm_node(state: ChatState) -> dict:
    response = model.invoke(state["messages"])
    return {"response": response.content, "messages": [response]}


def check_node(state: ChatState) -> Command:
    words = ["股票", "摄影", "天气", "金融"]
    for word in words:
        if word in state["response"]:
            feedback = HumanMessage(
                content=f"上一段文字包含不允许内容「{word}」，请避开该主题重新写"
            )
            # 不合格：Command 一步完成「更新 State（写回反馈）」+「指定下一跳（回 llm_node 重写）」
            return Command(goto="llm_node", update={"messages": [feedback]})
    # 合格：跳向 finish_node 收尾，无需再更新 State
    return Command(goto="finish_node")


def finish_node(state: ChatState) -> dict:
    return {}


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("check_node", check_node)
builder.add_node("finish_node", finish_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "check_node")
# check_node 没有出边：走向完全由它返回的 Command.goto 决定（第 2 节的等价写法是条件边）
builder.add_edge("finish_node", END)

graph = builder.compile()
display(graph)

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="请写一段简短文字，主题从金融、股票、摄影、天气、运动中选择一个"
            )
        ],
    }
)
rprint(result)